# SmartCart — Part 1: Data Preprocessing

Load and clean interaction + product metadata, build a user–item matrix (missing ratings as 0), and aggregate behavior by user and category.


## 1. Load data

In [1]:
import pandas as pd
from pathlib import Path


def resolve_data_dir() -> Path:
    for root in (Path.cwd(), *Path.cwd().parents):
        d = root / "data"
        if (d / "ecommerce_user_data.csv").exists() and (d / "product_details.csv").exists():
            return d
    raise FileNotFoundError(
        "Could not find data/ with ecommerce_user_data.csv and product_details.csv"
    )


DATA_DIR = resolve_data_dir()
user_data = pd.read_csv(DATA_DIR / "ecommerce_user_data.csv")
product_data = pd.read_csv(DATA_DIR / "product_details.csv")

print("Data directory:", DATA_DIR.resolve())
print("User interactions:", user_data.shape)
print("Products:", product_data.shape)
display(user_data.head())
display(product_data.head())

Data directory: /Users/youssef/Desktop/SOEN-471-A2/data
User interactions: (724, 5)
Products: (100, 3)


,UserID,ProductID,Rating,Timestamp,Category
0,U000,P0009,5,2024-09-08,Books
1,U000,P0020,1,2024-09-02,Home
2,U000,P0012,4,2024-10-18,Books
3,U000,P0013,1,2024-09-18,Clothing
4,U000,P0070,4,2024-09-16,Toys


,ProductID,ProductName,Category
0,P0000,Toys Item 0,Clothing
1,P0001,Clothing Item 1,Electronics
2,P0002,Books Item 2,Electronics
3,P0003,Clothing Item 3,Electronics
4,P0004,Clothing Item 4,Electronics


## 2. Clean data

- Parse `Timestamp` to datetime  
- Drop rows with missing `UserID`, `ProductID`, or `Rating`  
- Restrict to products that exist in `product_details`  
- Resolve duplicate `(UserID, ProductID)` by keeping the latest interaction

In [2]:
user_data["Timestamp"] = pd.to_datetime(user_data["Timestamp"], errors="coerce")

before = len(user_data)
user_data = user_data.dropna(subset=["UserID", "ProductID", "Rating"]).copy()
valid_products = set(product_data["ProductID"])
user_data = user_data[user_data["ProductID"].isin(valid_products)]

user_data = user_data.sort_values("Timestamp").drop_duplicates(
    subset=["UserID", "ProductID"], keep="last"
)

print(f"Rows before cleaning: {before}, after: {len(user_data)}")
user_data.info()

Rows before cleaning: 724, after: 724
<class 'pandas.DataFrame'>
Index: 724 entries, 464 to 247
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   UserID     724 non-null    str           
 1   ProductID  724 non-null    str           
 2   Rating     724 non-null    int64         
 3   Timestamp  724 non-null    datetime64[us]
 4   Category   724 non-null    str           
dtypes: datetime64[us](1), int64(1), str(3)
memory usage: 33.9 KB


## 3. Merge with product catalog

Attach `ProductName` and the catalog `Category` (from products). The interaction file’s `Category` is kept as `CategoryAtInteraction` for comparison.

In [3]:
user_merged = user_data.rename(columns={"Category": "CategoryAtInteraction"}).merge(
    product_data,
    on="ProductID",
    how="left",
    suffixes=("", "_dup"),
)

assert user_merged["ProductName"].notna().all(), "Unexpected products without metadata"
user_merged.head()

,UserID,ProductID,Rating,Timestamp,CategoryAtInteraction,ProductName,Category
0,U032,P0091,2,2024-09-01,Electronics,Clothing Item 91,Electronics
1,U034,P0089,1,2024-09-01,Beauty,Clothing Item 89,Beauty
2,U035,P0022,5,2024-09-01,Beauty,Toys Item 22,Beauty
3,U011,P0054,3,2024-09-01,Toys,Home Item 54,Toys
4,U018,P0073,2,2024-09-01,Beauty,Toys Item 73,Beauty


## 4. User–item matrix

Rows = users, columns = products, values = rating. Unrated pairs are **0** (not missing), as required for dense cosine similarity later.

In [4]:
user_item_matrix = user_merged.pivot_table(
    index="UserID", columns="ProductID", values="Rating", aggfunc="mean"
)
user_item_matrix_filled = user_item_matrix.fillna(0)

sparsity = 1.0 - (user_merged.shape[0] / (user_item_matrix.shape[0] * user_item_matrix.shape[1]))
print(f"Matrix shape: {user_item_matrix_filled.shape}")
print(f"Observed sparsity (1 - known ratings / all cells): {sparsity:.4f}")
user_item_matrix_filled.iloc[:5, :8]

Matrix shape: (50, 100)
Observed sparsity (1 - known ratings / all cells): 0.8552


ProductID,P0000,P0001,P0002,P0003,P0004,P0005,P0006,P0007
UserID,,,,,,,,
U000,0.0,0.0,0.0,3.0,0.0,5.0,0.0,3.0
U001,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0
U002,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0
U003,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
U004,0.0,3.0,0.0,0.0,0.0,0.0,2.0,0.0


## 5. Aggregate behavior by user and category

Uses catalog **Category** from the merged product table.

In [5]:
user_category_agg = (
    user_merged.groupby(["UserID", "Category"], as_index=False)
    .agg(TotalInteractions=("Rating", "count"), AverageRating=("Rating", "mean"))
)
user_category_agg.head(15)

,UserID,Category,TotalInteractions,AverageRating
0,U000,Books,6,3.666667
1,U000,Clothing,3,1.666667
2,U000,Electronics,3,3.666667
3,U000,Home,2,1.000000
4,U000,Toys,6,3.500000
5,U001,Beauty,1,4.000000
6,U001,Books,4,2.500000
7,U001,Clothing,4,2.500000
8,U001,Electronics,2,4.000000
9,U001,Home,1,2.000000
